# 01 — Exploratory Data Analysis: PlantVillage Dataset

**Dataset**: [abdallahalidev/plantvillage-dataset](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset) — Kaggle  
**Paper**: Mohanty et al. (2016) — *Using Deep Learning for Image-Based Plant Disease Detection*

This notebook covers:
1. Dataset download and structure exploration
2. Class distribution visualisation
3. Sample image grid per class
4. Colour vs segmented image comparison
5. Crop-level and disease-level breakdowns
6. Image size distribution

In [ ]:
# Install dependencies and download dataset
!pip install -q datasets torch torchvision timm opencv-python matplotlib seaborn tqdm
!pip install -q kaggle
!kaggle datasets download -d abdallahalidev/plantvillage-dataset --unzip -p /content/plantvillage

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from PIL import Image

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

DATA_COLOR = '/content/plantvillage/color'
DATA_SEG   = '/content/plantvillage/segmented'

print('Libraries loaded.')

## 1. Dataset structure

In [ ]:
# Discover class names from folder structure
class_names = sorted(os.listdir(DATA_COLOR))
num_classes = len(class_names)

print(f'Configs available : {os.listdir("/content/plantvillage")}')
print(f'Number of classes : {num_classes}')
print(f'\nFirst 5 classes   : {class_names[:5]}')
print(f'Last  5 classes   : {class_names[-5:]}')

# Parse crop and disease from class folder names (format: "Crop___Disease")
crops    = sorted(set(c.split('___')[0] for c in class_names))
diseases = sorted(set(c.split('___')[1] for c in class_names if '___' in c))
print(f'\nUnique crops    : {len(crops)}')
print(f'Unique diseases : {len(diseases)}')

# Count total images
total = sum(len(os.listdir(os.path.join(DATA_COLOR, c))) for c in class_names)
print(f'\nTotal images (color) : {total:,}')

## 2. Class distribution

In [ ]:
# Count images per class
counts = {c: len(os.listdir(os.path.join(DATA_COLOR, c))) for c in class_names}
labels_sorted = sorted(class_names, key=lambda c: -counts[c])
counts_sorted = [counts[c] for c in labels_sorted]

print(f'Most common  : {labels_sorted[0]} ({counts_sorted[0]:,} imgs)')
print(f'Least common : {labels_sorted[-1]} ({counts_sorted[-1]:,} imgs)')

fig, ax = plt.subplots(figsize=(13, 11))
colors  = plt.cm.tab20(np.linspace(0, 1, num_classes))
bars    = ax.barh(labels_sorted, counts_sorted, color=colors)
ax.set_xlabel('Number of images', fontsize=12)
ax.set_title('PlantVillage — Class Distribution', fontsize=13, fontweight='bold')
ax.bar_label(bars, padding=3, fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

## 3. Crop-level breakdown

In [ ]:
crop_counts = defaultdict(int)
for cls, cnt in counts.items():
    crop = cls.split('___')[0]
    crop_counts[crop] += cnt

crops_sorted = sorted(crop_counts, key=lambda c: -crop_counts[c])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(crops_sorted, [crop_counts[c] for c in crops_sorted], color='steelblue')
ax.set_xlabel('Crop')
ax.set_ylabel('Images')
ax.set_title('Images per crop species', fontsize=12, fontweight='bold')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

print('Images per crop:')
for crop in crops_sorted:
    print(f'  {crop:<40} {crop_counts[crop]:>6,}')

## 4. Sample image grid (8 classes × 4 images)

In [ ]:
N_CLASSES = 8
N_PER_CLS = 4

fig, axes = plt.subplots(N_CLASSES, N_PER_CLS, figsize=(N_PER_CLS * 3, N_CLASSES * 2.5))

for row, cls in enumerate(class_names[:N_CLASSES]):
    cls_dir = os.path.join(DATA_COLOR, cls)
    files   = sorted(os.listdir(cls_dir))[:N_PER_CLS]
    for col, fname in enumerate(files):
        ax  = axes[row][col]
        img = Image.open(os.path.join(cls_dir, fname)).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            short = cls.replace('___', '\n')
            ax.set_ylabel(short, fontsize=8, rotation=0, labelpad=90, va='center')

fig.suptitle('Sample Images per Class (first 8 classes)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_grid.png', bbox_inches='tight')
plt.show()

## 5. Colour vs Segmented comparison

In [ ]:
N_EXAMPLES = 6

color_samples, seg_samples, labels_ex = [], [], []

for cls in class_names[:N_EXAMPLES]:
    color_dir = os.path.join(DATA_COLOR, cls)
    seg_dir   = os.path.join(DATA_SEG,   cls)

    # Match by filename — same file appears in both configs
    shared_files = sorted(
        set(os.listdir(color_dir)) & set(os.listdir(seg_dir))
    )
    if not shared_files:
        shared_files = sorted(os.listdir(color_dir))

    fname = shared_files[0]
    color_samples.append(Image.open(os.path.join(color_dir, fname)).convert('RGB'))
    seg_samples.append(Image.open(os.path.join(seg_dir,   fname)).convert('RGB'))
    labels_ex.append(cls)

fig, axes = plt.subplots(2, N_EXAMPLES, figsize=(N_EXAMPLES * 3, 6))

for col in range(N_EXAMPLES):
    axes[0][col].imshow(color_samples[col])
    axes[0][col].axis('off')
    axes[0][col].set_title(labels_ex[col].split('___')[0], fontsize=8)

    axes[1][col].imshow(seg_samples[col])
    axes[1][col].axis('off')

axes[0][0].set_ylabel('Color',     fontsize=10, rotation=90)
axes[1][0].set_ylabel('Segmented', fontsize=10, rotation=90)

fig.suptitle('Colour vs Pre-Segmented Images', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('color_vs_segmented.png', bbox_inches='tight')
plt.show()

## 6. Image size distribution

In [ ]:
import random

# Collect all image paths
all_paths = []
for cls in class_names:
    cls_dir = os.path.join(DATA_COLOR, cls)
    all_paths.extend(os.path.join(cls_dir, f) for f in os.listdir(cls_dir))

sample_paths = random.sample(all_paths, min(500, len(all_paths)))

widths, heights = [], []
for p in sample_paths:
    with Image.open(p) as img:
        widths.append(img.width)
        heights.append(img.height)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.hist(widths,  bins=20, color='steelblue', edgecolor='white')
ax1.set_title('Width distribution (px)')
ax1.set_xlabel('Pixels')

ax2.hist(heights, bins=20, color='coral', edgecolor='white')
ax2.set_title('Height distribution (px)')
ax2.set_xlabel('Pixels')

plt.suptitle(f'Image Size Distribution (n={len(sample_paths)})', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Width  — mean: {np.mean(widths):.0f}  min: {min(widths)}  max: {max(widths)}')
print(f'Height — mean: {np.mean(heights):.0f}  min: {min(heights)}  max: {max(heights)}')

---
## Summary

| Metric | Value |
|--------|-------|
| Total images (color) | ~54,306 |
| Crop species | 14 |
| Classes (crop × condition) | 38 |
| Configs | color, grayscale, segmented |
| Train / Test split | Manual 80/20 (see notebook 02) |

**Next**: `02_classification.ipynb` — transfer learning with EfficientNet-B0.